# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## 1. Initialization

In [4]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [5]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
scale = 2
init_dask_cluster_eopf(scale=scale)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Get existing dask cluster: 'c74914864667465c96da4ca756e3a342'
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/c74914864667465c96da4ca756e3a342/status
Dask workers for 'dask-eopf' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [6]:
# Other imports
import getpass
import os
import os.path as osp
from importlib import reload
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

15:48:34.869 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

15:48:34.872 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

15:48:34.873 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

15:48:34.875 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

15:48:34.876 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

15:48:34.877 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

15:48:34.903 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0/config' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

In [7]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [23]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [9]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id '352dc420-a63a-488c-8c2f-0a09b2fc248a'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/352dc420-a63a-488c-8c2f-0a09b2fc248a


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [10]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


## 3. Run Prefect flow for S1 short data (~1 minute)

In [11]:
output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'


In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'versed-mustang'.
└── UUID: f7e7d451-bf0e-4a2b-9816-f41865a79f58
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-02-27 16:12:08 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/f7e7d451-bf0e-4a2b-9816-f41865a79f58
Watching flow run 'versed-mustang'...


16:12:08.913 | INFO    | prefect - Flow run is in state 'Scheduled'
16:12:13.936 | INFO    | prefect - Flow run is in state 'Pending'


In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 4. Run Prefect flow for S1 full data (~1 hour)

In [18]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1'


In [19]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'judicious-mammoth'.
└── UUID: f0eab441-c2a4-4ea5-81e2-b9edac845b70
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1'}
└── Job Variables: {}
└── Scheduled start time: 2025-02-26 17:07:27 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/f0eab441-c2a4-4ea5-81e2-b9edac845b70
Watching flow run 'judicious-mammoth'...


17:07:27.945 | INFO    | prefect - Flow run is in state 'Scheduled'
17:07:32.969 | INFO    | prefect - Flow run is in state 'Pending'


17:07:36.818 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:45257

17:07:37.205 | INFO    | Task run 'hack_payload-8b3' - Uploaded from '/tmp/tmp_6fgdj_o' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s1/S1A_20240410083700053369/.empty'.

17:07:37.215 | INFO    | Task run 'hack_payload-8b3' - Finished in state Completed()

...

17:08:33.160 | INFO    | prefect - Flow run is in state 'Running'
17:08:38.180 | INFO    | prefect - Flow run is in state 'Running'
17:08:43.194 | INFO    | prefect - Flow run is in state 'Running'
17:08:48.209 | INFO    | prefect - Flow run is in state 'Running'
17:08:53.224 | INFO    | prefect - Flow run is in state 'Running'
17:08:58.238 | INFO    | prefect - Flow run is in state 'Running'
17:09:03.254 | INFO    | prefect - Flow run is in state 'Running'
17:09:08.269 | INFO    | prefect - Flow run is in state 'Running'


17:09:12.144 | INFO    | l0.cadu_processing - loading packets from : rs-cluster-temp/stations/CADIP/S1A_20240410083700053369/ch_1/DCS_01_S1A_20240410083700053369_ch1_DSDB_00004.raw

17:09:13.294 | INFO    | prefect - Flow run is in state 'Running'
17:09:18.308 | INFO    | prefect - Flow run is in state 'Running'
17:09:23.326 | INFO    | prefect - Flow run is in state 'Running'
17:09:28.345 | INFO    | prefect - Flow run is in state 'Running'
17:09:33.365 | INFO    | prefect - Flow run is in state 'Running'
17:09:38.389 | INFO    | prefect - Flow run is in state 'Running'
17:09:43.407 | INFO    | prefect - Flow run is in state 'Running'
17:09:48.431 | INFO    | prefect - Flow run is in state 'Running'
17:09:53.447 | INFO    | prefect - Flow run is in state 'Running'
17:09:58.471 | INFO    | prefect - Flow run is in state 'Running'
17:10:03.495 | INFO    | prefect - Flow run is in state 'Running'
17:10:08.514 | INFO    | prefect - Flow run is in state 'Running'
17:10:13.534 | INFO    | prefect - Flow run is in state 'Running'
17:10:18.558 | INFO    | prefect - Flow run is in state 'Running'
17:10:23.591 | INFO    | prefect - Flow run is in state 'Running'
17:10:28.6

Flow run finished in state 'Failed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [7]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s3'


In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'impressive-lionfish'.
└── UUID: d4f2a581-1009-40a0-9a69-d917a7796193
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's3/s3_dordop_payload.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s3'}
└── Job Variables: {}
└── Scheduled start time: 2025-02-27 08:28:00 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/d4f2a581-1009-40a0-9a69-d917a7796193
Watching flow run 'impressive-lionfish'...


08:28:00.429 | INFO    | prefect - Flow run is in state 'Scheduled'
08:28:05.445 | INFO    | prefect - Flow run is in state 'Pending'
08:28:10.460 | INFO    | prefect - Flow run is in state 'Pending'


08:28:11.697 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:38127

08:28:12.217 | INFO    | Task run 'hack_payload-d88' - Uploaded from '/tmp/tmp43akx5lk' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s3/S3B_20240411142312031054/.empty'.

08:28:12.228 | INFO    | Task run 'hack_payload-d88' - Finished in state Completed()

...

08:58:32.436 | INFO    | Task run 'all_my_eopf_code-15a' - 2025-02-27 08:58:32,436 - l0.cadu_processing - INFO - Ending of the ISPExtractionUnit <=======

08:58:32.539 | INFO    | Task run 'all_my_eopf_code-15a' - 2025-02-27 08:58:32,539 - l0.cadu_processing - INFO - Starting of the ProcessingWindowUnit

08:58:32.540 | INFO    | Task run 'all_my_eopf_code-15a' - 2025-02-27 08:58:32,539 - l0.cadu_processing - INFO - Processing product navatt

08:58:32.656 | INFO    | Task run 'all_my_eopf_code-15a' - 2025-02-27 08:58:32,656 - l0.cadu_processing - INFO - Processing product olci

08:58:37.214 | INFO    | prefect - Flow run is in state 'Running'
08:58:42.228 | INFO    | prefect - Flow run is in state 'Running'
08:58:47.250 | INFO    | prefect - Flow run is in state 'Running'
08:58:52.264 | INFO    | prefect - Flow run is in state 'Running'
08:58:57.282 | INFO    | prefect - Flow run is in state 'Running'
08:59:02.298 | INFO    | prefect - Flow run is in state 'Running'
08:59:07.314 | INFO    | prefect - Flow run is in state 'Running'
08:59:12.332 | INFO    | prefect - Flow run is in state 'Running'
08:59:17.361 | INFO    | prefect - Flow run is in state 'Running'
08:59:22.378 | INFO    | prefect - Flow run is in state 'Running'
08:59:27.403 | INFO    | prefect - Flow run is in state 'Running'
08:59:32.419 | INFO    | prefect - Flow run is in state 'Running'
08:59:37.440 | INFO    | prefect - Flow run is in state 'Running'
08:59:42.462 | INFO    | prefect - Flow run is in state 'Running'
08:59:47.483 | INFO    | prefect - Flow run is in state 'Running'
08:59:52.5

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s3")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [ ]:
debug_flow = False

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=scale)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

In [ ]:
if debug_flow:
    import first_l0_processor
    reload(first_l0_processor)
    results = first_l0_processor.first_l0_processor(**s1_short)
    display(results)